This code has been adapated from [ICCM-25_RayD](https://github.com/THiCC-Lab/paper-repos-pub/tree/main/ICCM-25_RayD). The IAT model uses ACT-R integrated GUI for vision module. So ensure that you have ACT-R software downloaded from https://act-r.psy.cmu.edu/software/. Follow the steps on the website for setup and then start the ACT-R gui by clicking run-act-r.bat file (if running on windows machine).

In [ ]:
from hdm_actr import oscillators, hrr
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta, expon, geom, hypergeom
from math import isclose
from hdm_actr.oscillators import TimeVector
from hdm_actr.hrr import HRR
import os
import itertools
from tqdm.contrib.itertools import product as tqdm_product
import json
from datetime import datetime
import socket

Here we import the script which opens a connection to ACT-R with a HDM as its memory module. Make sure you have ACT-R GUI up and running before you run this.

In [ ]:
import hdm_actr.hdm_actr

If everything is set up correctly, above it should say "ACT-R connection has been started." 

Before running the ACT-R model ensure that the ACT-R console is pointing to the directory "SCAI_DulamDG\ACT-R models"

In [ ]:
# Load the model
hdm_actr.actr.load_act_r_code("hdm-iat-task.lisp")


In [ ]:
test = hdm_actr.hdm_actr.hdm_modules['iatmodel']
print("before",test.chunk_count)


The file that has chunks generated from LLM is sent as input.

In [ ]:
import json

def has_json(text: str) -> bool:
    start = text.find("{")
    if start == -1:
        return False

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return True

    return False

# Post Processing: If sentence starts with Solution and has double curly braces, do post processing to extract JSON
def invalid_sentence_to_json_ready(sentence):
    print("Invalid Original Sentence: ", sentence[:300])
    
    if not has_json(sentence):
        print("No JSON found in sentence.")
        return ""
    
    # Fix templated braces first
    text = sentence.replace("{{", "{").replace("}}", "}")

    start = text.find("{")
    if start == -1:
        raise ValueError("No JSON object start found")

    depth = 0
    end = None

    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                end = i + 1
                break

    if end is None:
        raise ValueError("No complete JSON object found")

    return text[start:end].strip()

def llm_sentence_to_chunk_args(sentence):
    # print("LLM Sentence: ", sentence[:300])

    # Post Processing: If sentence is empty string, return None or empty list
    if not sentence:
        print("Empty LLM output, skipping...")
        return None

    try:
        data = json.loads(sentence)
    except json.JSONDecodeError as e:
        json_ready_sentence = invalid_sentence_to_json_ready(sentence)
        
        # if returns empty string, skip
        if not json_ready_sentence:
            print("No JSON could be extracted, skipping...")
            return None
        
        print("Processed LLM Sentence for JSON parsing: ", json_ready_sentence[:300])
        data = json.loads(json_ready_sentence)
    
    chunk_args = ["ISA", data["isa"]]

    for slot, value in data["slots"].items():
        chunk_args.extend([slot, value])

    # print("Chunk Args: ", chunk_args)
    
    return chunk_args

def llm_sentence_to_chunk(sentence, hdm):
    # print("LLM Sentence: ", sentence[:300])
    
    # str to dict
    chunk_args = llm_sentence_to_chunk_args(sentence)
    
    if chunk_args is None:
        return None
    
    # lets make a name for the new chunk
    chunk_name = f"{chunk_args[1]}-chunk-{hdm.chunk_count}"
    # print("Chunk Name: ", chunk_name)
    
    chunk = hdm_actr.hdm_actr.Chunk(chunk_args, 
                                    chunk_name=chunk_name, 
                                    has_slots=True,
                                    check_defined=False,
                                    store_chunk=False)

    hdm.lock.acquire()
    hdm.add_chunk_to_dm(chunk)
    hdm.lock.release()
    
    return chunk


print("before",test.chunk_count)
with open("../Data/Inputs/LLM/all_llm_chunk.txt", "r",encoding="utf-16") as f:
        llm_outputs = f.read().split("\n")
        
for s in llm_outputs:
    # print("Num of chunk in DM:", hdm.chunk_count)
    print("Creating new chunk from LLM output...")
    chunk = llm_sentence_to_chunk(s, test)
    # print("Chunk: ", chunk)
print("after",test.chunk_count)



In [ ]:
test.hrr_mat(chunk_count=test.chunk_count)
# This calls the "run-once" method in ACT-R task.lisp file
hdm_actr.actr.call_command("run-once")